<a href="https://colab.research.google.com/github/YamenObeidat/Deep-learning-part-1/blob/main/DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

csv_path = (
    "Used_Car_Price_Prediction.csv"
    if os.path.exists("Used_Car_Price_Prediction.csv")
    else os.path.join("data", "Used_Car_Price_Prediction.csv")
)

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget (sale_price) summary:\n", df["sale_price"].describe())
df.head(3)

Using device: cpu
Shape: (7400, 29)

Missing values per column:
 body_type               103
transmission            556
registered_city          10
registered_state         10
source                  126
car_availability        620
original_price         3280
car_rating                9
ad_created_on             1
fitness_certificate       8
dtype: int64

Target (sale_price) summary:
 count       7,400.00
mean      454,889.19
std       282,702.33
min             0.00
25%       281,174.00
50%       382,449.00
75%       540,149.00
max     3,866,000.00
Name: sale_price, dtype: float64


,car_name,yr_mfr,fuel_type,kms_run,sale_price,city,times_viewed,body_type,transmission,variant,...,total_owners,broker_quote,original_price,car_rating,ad_created_on,fitness_certificate,emi_starts_from,booking_down_pymnt,reserved,warranty_avail
0,maruti swift,2015,petrol,8063,386399,noida,18715,hatchback,manual,lxi opt,...,2,397677,"404,177.00",great,2021-04-04T07:09:18.583,True,8975,57960,False,False
1,maruti alto 800,2016,petrol,23104,265499,noida,2676,hatchback,manual,lxi,...,1,272935,"354,313.00",great,2021-03-22T14:07:32.833,True,6167,39825,False,False
2,hyundai grand i10,2017,petrol,23402,477699,noida,609,hatchback,manual,sports 1.2 vtvt,...,1,469605,NaN,great,2021-03-20T05:36:31.311,True,11096,71655,False,False


In [5]:

df_clean = df.drop_duplicates().copy()
df_clean = df_clean[df_clean["sale_price"].notna() & (df_clean["sale_price"] > 0)].copy()
print("Rows after cleaning:", len(df_clean))  # Must be between 1,000 and 200,000

target_col = "sale_price"


cols_to_drop = [
    target_col,
    "broker_quote",
    "emi_starts_from",
    "booking_down_pymnt",
    "car_name",
    "variant",
    "rto",
    "registered_city",
    "ad_created_on",
]

X_df = df_clean.drop(columns=cols_to_drop).copy()
y = df_clean[target_col].to_numpy(dtype=np.float32).reshape(-1, 1)


num_cols = X_df.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]

for col in X_df.select_dtypes(include=["bool"]).columns:
    X_df[col] = X_df[col].astype(np.float32)

X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    X_df, y, test_size=0.30, random_state=42
)
X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df, y_temp, test_size=0.50, random_state=42
)

num_imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_num = scaler.fit_transform(num_imputer.fit_transform(X_train_df[num_cols]))
X_val_num = scaler.transform(num_imputer.transform(X_val_df[num_cols]))
X_test_num = scaler.transform(num_imputer.transform(X_test_df[num_cols]))

cat_imputer = SimpleImputer(strategy="most_frequent")
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat = encoder.fit_transform(cat_imputer.fit_transform(X_train_df[cat_cols]))
X_val_cat = encoder.transform(cat_imputer.transform(X_val_df[cat_cols]))
X_test_cat = encoder.transform(cat_imputer.transform(X_test_df[cat_cols]))

X_train = np.hstack([X_train_num, X_train_cat]).astype(np.float32)
X_val = np.hstack([X_val_num, X_val_cat]).astype(np.float32)
X_test = np.hstack([X_test_num, X_test_cat]).astype(np.float32)

print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:  ", X_val.shape, y_val.shape)
print("Test shape: ", X_test.shape, y_test.shape)

X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_val_t = torch.tensor(X_val)
y_val_t = torch.tensor(y_val)

Rows after cleaning: 7396
Train shape: (5177, 264) (5177, 1)
Val shape:   (1109, 264) (1109, 1)
Test shape:  (1110, 264) (1110, 1)
